# Navegação AR urbana — projeção da rota GPS sobre a imagem do motorista

**PIBIC 26/27** · protótipo do workflow completo

Este notebook demonstra, etapa por etapa, o pipeline de geometria projetiva que
leva uma rota gravada por GPS até uma faixa desenhada sobre o asfalto na
perspectiva do motorista.

| # | Módulo | Arquivo | O que faz |
|---|--------|---------|-----------|
| 1 | GPS | `modulos/gps.py` | GPX → ENU → suavização → heading (**Passo 1**) |
| 2 | Sincronização | `modulos/video.py`, `sync.py` | frame ↔ posição, rota futura no referencial do veículo |
| 3 | Calibração | `modulos/calibration.py` | `K`, distorção e pose da câmera (**Passos 2 e 3**) |
| 4 | Segmentação | `modulos/segmentation.py` | YOLOPv2: área dirigível, faixas, obstáculos |
| 5 | IPM / BEV | `modulos/ipm.py` | bird's-eye view e ajuste da rota à via visível |
| 6 | Projeção e render | `modulos/projection.py`, `render.py` | volta para a imagem, spline e oclusão (**Passo 4**) |

### Convenções de eixos (valem para todo o pipeline)

| Referencial | X | Y | Z |
|---|---|---|---|
| Veículo (origem no chão, sob a câmera) | frente | esquerda | cima |
| Câmera (OpenCV) | direita | baixo | frente |
| ENU (local) | leste | norte | cima |

`heading = 0` aponta para o Norte e cresce no sentido horário, como numa bússola.

---
## 0 · Ambiente e configuração

In [ ]:
import sys, subprocess
from pathlib import Path

# --- dependências -----------------------------------------------------
for modulo, pacote in {
    "cv2": "opencv-python", "numpy": "numpy", "pandas": "pandas",
    "matplotlib": "matplotlib", "scipy": "scipy",
    "gpxpy": "gpxpy", "pyproj": "pyproj", "torch": "torch",
}.items():
    try:
        __import__(modulo)
    except ImportError:
        print(f"instalando {pacote}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

# --- raiz do projeto no path -----------------------------------------
RAIZ = Path.cwd()
if not (RAIZ / "modulos").exists():
    RAIZ = RAIZ.parent            # rodando de dentro de notebooks/
sys.path.insert(0, str(RAIZ))

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (12, 7)

# Exibe uma imagem BGR do OpenCV no matplotlib
def mostrar(img, titulo="", eixo=False, tamanho=(9, 12)):
    plt.figure(figsize=tamanho)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img,
               cmap=None if img.ndim == 3 else "gray")
    plt.title(titulo)
    plt.axis("on" if eixo else "off")
    plt.tight_layout(); plt.show()

print("raiz:", RAIZ)

In [ ]:
from modulos.config import Config

cfg = Config(
    raiz=RAIZ,
    nome_percurso="volta_menor",

    # sincronização -----------------------------------------------------
    frame_alvo=692,
    offset_sync_s=-4.0,
    horizonte_s=12.0,
    distancia_max_m=40.0,

    # pose da câmera (ajuste manual — veja a seção 3.3) -------------------
    altura_camera_m=1.50,
    pitch_deg=-10.0,
    yaw_deg=0.0,
    roll_deg=0.0,

    # ajuste da rota à via ----------------------------------------------
    peso_centro=0.35,
    margem_borda_m=0.8,
    largura_faixa_ar_m=1.8,
)

print(cfg.resumo())
print()
for nome, existe in cfg.checar_arquivos().items():
    print(f"  {'OK ' if existe else 'FALTA'}  {nome}")

---
## Módulo 1 · Leitura e tratamento dos dados de GPS

> **Passo 1 do plano** — converter as coordenadas esféricas para um plano
> cartesiano local, onde a geometria projetiva passa a ser linear.

A conversão segue o caminho geodésico → **ECEF** → **ENU**: o ponto é levado
para um referencial cartesiano centrado na Terra e depois rotacionado para o
plano tangente à origem local (o primeiro ponto da trilha).

Duas decisões de projeto vale destacar:

* **o `heading` gravado pelo aplicativo é descartado** — ele vem rotacionado e
  ruidoso. O rumo é recalculado a partir da derivada temporal do ENU já
  suavizado, com *unwrap* antes da suavização para não haver salto em 0°/360°;
* **suavização Savitzky-Golay** antes de derivar: o GPS de celular oscila
  metros entre amostras, e derivar o sinal cru amplifica esse ruído.

In [ ]:
from modulos import gps

trilha = gps.preparar(cfg.gpx, cfg.janela_savgol, cfg.ordem_savgol)

print(gps.resumo(trilha))
trilha.head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))

ax[0].plot(trilha["east"], trilha["north"], "-", lw=1.5, color="#3366cc")
ax[0].scatter(trilha["east"].iloc[0], trilha["north"].iloc[0], c="g", s=60, label="início", zorder=5)
ax[0].scatter(trilha["east"].iloc[-1], trilha["north"].iloc[-1], c="r", s=60, label="fim", zorder=5)
ax[0].set_aspect("equal"); ax[0].grid(alpha=.3); ax[0].legend()
ax[0].set_title("Trajeto em ENU"); ax[0].set_xlabel("east (m)"); ax[0].set_ylabel("north (m)")

ax[1].plot(trilha["t"], np.rad2deg(trilha["heading"]), lw=1.2, color="#cc5500")
ax[1].grid(alpha=.3); ax[1].set_title("Heading recalculado")
ax[1].set_xlabel("t (s)"); ax[1].set_ylabel("graus (0 = Norte)")

ax[2].plot(trilha["t"], trilha["velocidade"] * 3.6, lw=1.2, color="#118844")
ax[2].grid(alpha=.3); ax[2].set_title("Velocidade")
ax[2].set_xlabel("t (s)"); ax[2].set_ylabel("km/h")

plt.tight_layout(); plt.show()

---
## Módulo 2 · Vídeo, sincronização e rota futura

O GPX e o vídeo têm relógios independentes. `offset_sync_s` alinha os dois:

$$t_{\text{vídeo}} = t_{\text{gpx}} + \text{offset}$$

Com o alinhamento em mãos, dois produtos saem daqui:

1. **onde o carro estava** no instante do frame (posição e heading interpolados);
2. **por onde ele vai passar**, reamostrado a cada 0,5 m. Reamostrar por
   *distância* — e não por tempo — é o que garante resolução uniforme da
   curva: a 50 km/h o GPS grava um ponto a cada ~25 m, esparso demais para
   projetar.

Por fim a rota é reescrita no referencial do veículo. Com o heading medido a
partir do Norte no sentido horário, os versores da base são
$f = (\sin h, \cos h)$ e $l = (-\cos h, \sin h)$, logo:

$$X = \Delta e \sin h + \Delta n \cos h \qquad Y = -\Delta e \cos h + \Delta n \sin h$$

Esses são exatamente os pontos $\mathbf{X}_W$ do plano.

In [ ]:
from modulos import video, sync

info = video.info(cfg.video)
print(info)
print()
print(sync.diagnosticar(trilha, info, cfg.offset_sync_s))

In [ ]:
estado = sync.estado_no_frame(trilha, cfg.frame_alvo, info, cfg.offset_sync_s)
print(estado)

rota_enu = sync.trajetoria_futura(
    trilha, estado, cfg.offset_sync_s, cfg.horizonte_s, cfg.distancia_max_m
)

rota_veiculo = sync.apenas_a_frente(
    sync.enu_para_veiculo(rota_enu, estado, cfg.usar_elevacao)
)

print(f"\nwaypoints futuros: {len(rota_veiculo)}")
print(f"alcance          : {rota_veiculo[:, 0].max():.1f} m à frente")
print(f"desvio lateral   : {rota_veiculo[:, 1].min():+.2f} m a {rota_veiculo[:, 1].max():+.2f} m")

frame = video.extrair_frame(cfg.video, cfg.frame_alvo)
mostrar(frame, f"frame {cfg.frame_alvo}")

In [ ]:
plt.figure(figsize=(6, 9))
plt.plot(rota_veiculo[:, 1], rota_veiculo[:, 0], lw=2.5, color="#cc3300")
plt.scatter([0], [0], marker="s", s=120, c="k", label="veículo", zorder=5)
plt.gca().invert_xaxis()          # Y positivo = esquerda -> fica à esquerda
plt.gca().set_aspect("equal")
plt.grid(alpha=.3); plt.legend()
plt.title("Rota futura no referencial do veículo")
plt.xlabel("Y — esquerda (m)"); plt.ylabel("X — frente (m)")
plt.tight_layout(); plt.show()

---
## Módulo 3 · Calibração: intrínsecos e extrínsecos

> **Passos 2 e 3 do plano.**

### 3.1 Intrínsecos

$\mathbf{K}$ e os coeficientes de distorção vêm do `calibracao_camera.json`
(tabuleiro de xadrez). Como as fotos de calibração têm resolução diferente da
do vídeo, $\mathbf{K}$ precisa ser reescalado: $f$ e $(u_0, v_0)$ escalam
linearmente, enquanto os coeficientes de distorção são adimensionais e não
mudam.

O frame é então **corrigido** (`cv2.undistort`). A partir daí a imagem obedece
ao modelo pinhole puro — o que faz a projeção e a IPM serem exatamente
inversas uma da outra.

### 3.2 Extrínsecos

⚠️ Os `rvec`/`tvec` gravados no JSON descrevem a pose de **cada tabuleiro**
em relação à câmera; não servem para dizer onde a câmera está montada no
carro. A pose de montagem é declarada explicitamente:

$$\mathbf{R} = R_z(\text{roll}) \, R_x(-\text{pitch}) \, R_y(\text{yaw}) \, R_{\text{base}},
\qquad \mathbf{t} = -\mathbf{R}\,\mathbf{C}, \quad \mathbf{C} = (0, 0, h)^T$$

onde $R_{\text{base}}$ apenas realinha a base do veículo (X frente, Y esquerda,
Z cima) com a da câmera OpenCV (X direita, Y baixo, Z frente).

### 3.3 Projeção

$$\mathbf{X}_C = \mathbf{R}\,\mathbf{X}_W + \mathbf{t}
\qquad\longrightarrow\qquad
\tilde{\mathbf{y}} = \mathbf{K}\,\mathbf{X}_C
\qquad\longrightarrow\qquad
u = \frac{\tilde y_0}{\tilde y_2},\; v = \frac{\tilde y_1}{\tilde y_2}$$

In [ ]:
from modulos import calibration as cal

intr, pose = cal.preparar(
    cfg.arquivo_calibracao, (frame.shape[1], frame.shape[0]),
    cfg.altura_camera_m, cfg.pitch_deg, cfg.yaw_deg, cfg.roll_deg,
)

print(intr)
print(pose)
print("\nR (veículo -> câmera):\n", np.round(pose.R, 4))
print("\nt:", np.round(pose.t.ravel(), 3))
print("\nP = K·[R|t]:\n", np.round(cal.matriz_projecao(intr, pose), 1))

frame_corr = cal.corrigir_distorcao(frame, intr)
v_horizonte = cal.linha_do_horizonte(intr, pose)
print(f"\nhorizonte em v = {v_horizonte:.0f} px (de {frame.shape[0]})")

### Conferência da pose: grade métrica projetada no asfalto

Este é o teste que valida `altura`, `pitch` e `yaw`. A grade abaixo marca
linhas a cada 5 m e ±1,75 m de largura (uma faixa típica). **Ajuste os
parâmetros no `cfg` até que as linhas assentem sobre o asfalto real** — em
especial, a linha do horizonte deve coincidir com o horizonte da cena.

In [ ]:
from modulos import projection as proj
from modulos import render

grade_img = frame_corr.copy()

for d in range(5, 45, 5):                       # linhas transversais
    pts = np.column_stack([np.full(21, d), np.linspace(-5, 5, 21), np.zeros(21)])
    uv = proj.projetar_pinhole(pts, intr, pose).uv
    ok = np.isfinite(uv).all(axis=1)
    if ok.sum() > 1:
        cv2.polylines(grade_img, [np.int32(uv[ok]).reshape(-1, 1, 2)],
                      False, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(grade_img, f"{d}m", tuple(np.int32(uv[ok][-1]) + [8, -8]),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2, cv2.LINE_AA)

for y in (-1.75, 0.0, 1.75):                    # linhas longitudinais
    pts = np.column_stack([np.linspace(2, 40, 60), np.full(60, y), np.zeros(60)])
    uv = proj.projetar_pinhole(pts, intr, pose).uv
    ok = np.isfinite(uv).all(axis=1)
    cor = (0, 255, 0) if y == 0 else (255, 150, 0)
    cv2.polylines(grade_img, [np.int32(uv[ok]).reshape(-1, 1, 2)],
                  False, cor, 2, cv2.LINE_AA)

grade_img = render.desenhar_horizonte(grade_img, v_horizonte)
mostrar(grade_img, f"Grade métrica — h={cfg.altura_camera_m} m, pitch={cfg.pitch_deg}°")

#### Atalho para acertar o pitch

O horizonte é a projeção de um ponto no infinito, então ele depende **só** do
pitch — a altura da câmera não entra:

$$v_{\text{horizonte}} = c_y - f_y \tan(-\text{pitch})$$

Em vez de tentar ângulos às cegas, localize na imagem a linha onde o solo
encontra o céu e inverta a equação. Para a altura, o análogo é apontar um
objeto no asfalto cuja distância você conheça.

> **Observação sobre este frame:** com `pitch = -10°` a grade cai acima do
> asfalto visível. O celular estava apoiado no painel, apontando levemente
> para cima — valores entre `0°` e `+5°` assentam melhor nesta cena. Use as
> células abaixo para calibrar e depois atualize o `cfg`.

In [ ]:
# 1) leia na imagem acima a linha v onde o solo encontra o céu
V_HORIZONTE_OBSERVADO = 780      # <-- ajuste

pitch_sugerido = cal.pitch_pelo_horizonte(intr, V_HORIZONTE_OBSERVADO)
print(f"pitch sugerido pelo horizonte: {pitch_sugerido:+.2f}°  (atual: {cfg.pitch_deg:+.1f}°)")

# 2) opcional: um ponto no asfalto de distância conhecida -> altura
# print(cal.altura_por_referencia(intr, pitch_sugerido, v=1350, distancia_m=8.0))

# 3) comparação lado a lado de vários pitches
def grade_com_pitch(pitch, altura=cfg.altura_camera_m):
    i, p = cal.preparar(cfg.arquivo_calibracao, (frame.shape[1], frame.shape[0]), altura, pitch)
    img = cal.corrigir_distorcao(frame, i)
    for d in range(5, 45, 5):
        pts = np.column_stack([np.full(21, d), np.linspace(-5, 5, 21), np.zeros(21)])
        uv = proj.projetar_pinhole(pts, i, p).uv; ok = np.isfinite(uv).all(axis=1)
        if ok.sum() > 1:
            cv2.polylines(img, [np.int32(uv[ok]).reshape(-1, 1, 2)], False, (0, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(img, f"{d}m", tuple(np.int32(uv[ok][-1]) + [8, -8]),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 255), 3, cv2.LINE_AA)
    for y in (-1.75, 0.0, 1.75):
        pts = np.column_stack([np.linspace(2, 40, 80), np.full(80, y), np.zeros(80)])
        uv = proj.projetar_pinhole(pts, i, p).uv; ok = np.isfinite(uv).all(axis=1)
        cv2.polylines(img, [np.int32(uv[ok]).reshape(-1, 1, 2)], False,
                      (0, 255, 0) if y == 0 else (255, 150, 0), 3, cv2.LINE_AA)
    return render.desenhar_horizonte(img, cal.linha_do_horizonte(i, p))

mostrar(render.montar_painel(
    {f"pitch {p:+.0f}": grade_com_pitch(p) for p in (-10, -5, 0, 5)}, largura_col=400
), "Qual pitch assenta a grade no asfalto?", tamanho=(16, 8))

In [ ]:
# Escala vertical da imagem: quantos metros à frente cada linha representa
linhas = np.linspace(int(v_horizonte) + 60, frame.shape[0] - 1, 12).astype(int)
print(f"{'v (px)':>8} {'distância':>12}")
for v in linhas:
    print(f"{v:>8} {cal.distancia_no_solo(intr, pose, v):>10.1f} m")

---
## Módulo 4 · Segmentação da via com YOLOPv2

O modelo entrega três saídas num único *forward*: detecção de veículos,
**área dirigível** e **faixas**. Duas delas alimentam etapas seguintes:

* a área dirigível define o corredor a que a rota será ajustada (módulo 5);
* as caixas de veículos truncam a rota por oclusão (módulo 6).

O pré-processamento usa **letterbox** (redimensiona preservando proporção e
completa com bordas) em vez de um `resize` direto. O padding é guardado e
desfeito ao voltar a máscara para a resolução original — sem isso a máscara
sairia esticada e desalinhada do asfalto.

In [ ]:
from modulos import segmentation as seg_mod

yolop = seg_mod.carregar_modelo(cfg.pesos_yolop, cfg.repo_yolop)
print("device:", yolop.device)

seg = seg_mod.segmentar(
    yolop, frame_corr, cfg.tamanho_inferencia,
    cfg.limiar_area_dirigivel, cfg.limiar_faixas,
)
seg.area_dirigivel = seg_mod.limpar_mascara(seg.area_dirigivel)

print(seg)
mostrar(seg_mod.sobrepor(frame_corr, seg), "YOLOPv2 — área dirigível (verde), faixas (azul), veículos (amarelo)")

---
## Módulo 5 · Bird's-eye view (IPM) e ajuste da rota à via

> O "truque" do plano: em vez de projetar a rota direto na foto, primeiro
> levamos **a imagem** para uma vista superior métrica.

Para pontos do plano do solo ($Z = 0$) a terceira coluna de $\mathbf{R}$
desaparece e a projeção colapsa numa homografia:

$$\mathbf{H} = \mathbf{K}\,[\,\mathbf{r}_1 \;|\; \mathbf{r}_2 \;|\; \mathbf{t}\,]$$

Invertê-la *é* o mapeamento de perspectiva inversa. Na BEV, rota do GPS e área
dirigível vivem no mesmo sistema métrico, e o casamento entre as duas fica
trivial:

1. **corredor dirigível** — para cada distância à frente, mede-se a extensão
   lateral do asfalto conectada à posição do carro (evitando confundir a via
   da mão contrária com a atual);
2. **atração ao centro** (`peso_centro`) — absorve o erro sistemático de
   poucos metros do GPS de celular, que costuma jogar a rota sobre a calçada;
3. **confinamento** — a rota é truncada para ficar a pelo menos
   `margem_borda_m` das bordas detectadas;
4. **spline** no plano métrico — a suavização fica isotrópica, sem o artefato
   de curva certinha perto e serrilhada longe.

Este é o passo que materializa a premissa do projeto: *o espaço visível da rua
faz parte do trajeto do GPS*.

⚠️ A IPM só vale sob **hipótese de solo plano**: subidas e lombadas introduzem
erro proporcional à distância.

In [ ]:
from modulos import ipm

H_solo = cal.homografia_solo_para_imagem(intr, pose)
grade_bev = ipm.GradeBEV(cfg.bev_x_min, cfg.bev_x_max, cfg.bev_y_meia_largura, cfg.bev_px_por_m)

bev = ipm.para_bev(frame_corr, H_solo, grade_bev)
mascara_bev = ipm.mascara_para_bev(seg.area_dirigivel, H_solo, grade_bev)
corredor = ipm.extrair_corredor(mascara_bev, grade_bev)

print(f"BEV: {grade_bev.tamanho[0]}x{grade_bev.tamanho[1]} px "
      f"({grade_bev.px_por_m:.0f} px/m, {grade_bev.x_min:.0f}–{grade_bev.x_max:.0f} m)")
print(f"alcance visível da via: {corredor.alcance_visivel:.1f} m")
larg = corredor.largura[corredor.valido]
if len(larg):
    print(f"largura do corredor   : mediana {np.median(larg):.1f} m")

In [ ]:
rota_ajustada = ipm.ajustar_rota_ao_corredor(
    rota_veiculo, corredor, cfg.margem_borda_m, cfg.peso_centro
)
rota_ajustada = ipm.suavizar_rota(rota_ajustada)

desloc = np.interp(rota_ajustada[:, 0], rota_veiculo[:, 0], rota_veiculo[:, 1]) - rota_ajustada[:, 1]
print(f"correção lateral aplicada: média {np.abs(desloc).mean():.2f} m | máx {np.abs(desloc).max():.2f} m")

bev_anotada = ipm.desenhar_bev(bev, grade_bev, rota_veiculo, rota_ajustada, corredor)

fig, ax = plt.subplots(1, 2, figsize=(13, 8))
ax[0].imshow(mascara_bev, cmap="gray"); ax[0].set_title("Área dirigível em BEV"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(bev_anotada, cv2.COLOR_BGR2RGB))
ax[1].set_title("BEV — GPS (laranja) vs. ajustada (verde)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

---
## Módulo 6 · Projeção de volta e renderização

> **Passo 4 do plano.**

A rota corrigida volta para a imagem pela mesma cadeia $\mathbf{K}[\mathbf{R}|\mathbf{t}]$
e passa por três critérios de corte:

* **profundidade** — pontos com $z_c \le 0$ estão atrás da câmera; projetá-los
  produziria uma imagem espelhada no céu;
* **área dirigível** — o ponto precisa cair sobre asfalto segmentado;
* **oclusão** — pontos dentro da região inferior de uma caixa de veículo estão
  escondidos atrás dele.

A rota é então **truncada no primeiro corte**: uma linha que some e reaparece
depois de um obstáculo confunde mais do que ajuda.

A fita tem largura definida **em metros, no mundo** — não em pixels. Ao ser
projetada ela afina sozinha com a distância, que é o que dá a sensação de estar
colada no asfalto. Os quadriláteros são pintados do mais distante para o mais
próximo, respeitando a ordem de profundidade.

In [ ]:
rota_final, projecao = proj.filtrar_rota(
    rota_ajustada, intr, pose,
    mascara_via=seg.area_dirigivel,
    obstaculos=seg.obstaculos,
    exigir_via=True,
)

print(f"waypoints antes do corte : {len(rota_ajustada)}")
print(f"waypoints desenháveis    : {len(rota_final)}")
if len(rota_final):
    print(f"rota visível até         : {rota_final[:, 0].max():.1f} m")

In [ ]:
imagem = render.desenhar_rota(frame_corr, rota_final, intr, pose, cfg.largura_faixa_ar_m)
imagem = render.desenhar_marcadores_distancia(imagem, rota_final, intr, pose)
imagem = render.desenhar_minimapa(imagem, trilha, rota_enu, estado)
imagem = render.desenhar_hud(imagem, [
    f"frame {cfg.frame_alvo}  t={estado.t_video:.1f}s",
    f"heading {estado.heading_deg:.0f} deg   v {estado.velocidade*3.6:.0f} km/h",
    f"cam h={cfg.altura_camera_m:.2f}m pitch={cfg.pitch_deg:+.0f} deg",
    f"rota visivel: {rota_final[:,0].max():.0f} m" if len(rota_final) else "rota visivel: --",
])

mostrar(imagem, "Resultado — rota projetada sobre o asfalto", tamanho=(10, 16))

saida = cfg.saida / f"frame{cfg.frame_alvo:06d}_final.png"
cv2.imwrite(str(saida), imagem)
print("salvo em:", saida)

---
## 7 · Pipeline completo e processamento em lote

Tudo o que foi feito acima está encapsulado em `modulos/pipeline.py`. Útil para
varrer vários frames sem recarregar o modelo a cada iteração.

In [ ]:
from modulos import pipeline

resultado = pipeline.executar(cfg, yolop_carregado=yolop)
arquivos = resultado.salvar()

for nome, caminho in arquivos.items():
    print(f"  {nome:14s} {caminho.name}")

mostrar(render.montar_painel({
    "1. frame corrigido": resultado.frame_corrigido,
    "2. segmentação": seg_mod.sobrepor(resultado.frame_corrigido, resultado.seg),
    "3. resultado": resultado.imagem_final,
}, largura_col=420), "Workflow", tamanho=(16, 10))

In [ ]:
# Vários frames de uma vez (descomente para rodar)
# from dataclasses import replace
# for f in [400, 692, 1200, 2000]:
#     r = pipeline.executar(replace(cfg, frame_alvo=f), yolop_carregado=yolop, verboso=False)
#     r.salvar()
#     mostrar(r.imagem_final, f"frame {f}", tamanho=(7, 12))

---
## 8 · Limitações conhecidas e próximos passos

**Limitações desta versão**

1. **Pose da câmera é manual.** Altura e pitch são parâmetros ajustados a olho.
   O caminho natural é estimá-los pelo ponto de fuga das faixas detectadas pelo
   YOLOPv2, que dá pitch e roll diretamente; a altura continuaria como única
   entrada manual.
2. **Hipótese de solo plano.** A IPM assume $Z = 0$; em ladeira o erro cresce
   com a distância. A elevação do GPX existe (`cfg.usar_elevacao`) mas é
   ruidosa demais para uso direto.
3. **Sincronização por offset constante.** Não há correção de deriva entre os
   relógios ao longo do vídeo.
4. **Aspecto da calibração.** As fotos do tabuleiro e o vídeo têm proporções
   ligeiramente diferentes, então $f_x$ e $f_y$ recebem fatores de escala
   distintos. Refazer a calibração gravando **vídeo** em vez de fotos elimina
   essa aproximação.
5. **Um frame por vez.** Não há filtro temporal; em sequência, a faixa pode
   tremer entre frames.

**Próximos passos**

* estimativa automática de pitch/roll pelo ponto de fuga;
* filtro de Kalman sobre a pose e sobre a rota ajustada, para estabilizar o
  vídeo completo;
* comparação com a linha de base da arquitetura GAN-LSTM (`sensors-25-00820.pdf`);
* validação quantitativa: erro lateral entre a rota projetada e o eixo real da
  via, medido em BEV.